In [1]:
!pip install pytorch-fid
!pip install scikit-learn torchmetrics scipy
!pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.0 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5

In [2]:
# <<< THAY ĐỔI: Chạy dòng này trước tiên để cài đặt thư viện cần thiết
# !pip install lpips

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import umap
from itertools import product
import lpips # <<< THAY ĐỔI: Import thư viện lpips

# ==============================================================================
# 1. Cấu hình và Thiết lập (Configuration and Setup)
# ==============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "results_wae_annealing_v4_lpips" # <<< THAY ĐỔI: Đổi tên thư mục kết quả
os.makedirs(SAVE_DIR, exist_ok=True)

config = {
    "latent_dim": 32,
    "n_classes": 10,
    "batch_size": 128,
    "epochs": 50,
    "lr": 1e-3,
    "rho_prior": 0.7,
    "epsilon": 1e-8,
    # --- Cấu hình cho Loss ---
    "bce_weight": 0.3,      # <<< THAY ĐỔI: Trọng số cho BCE loss
    "lpips_weight": 0.7,    # <<< THAY ĐỔI: Trọng số cho LPIPS loss
    # --- Cấu hình cho Annealing ---
    "sup_mmd_weight": 20.0,
    "unsup_mmd_weight": 50.0,
    "anneal_epochs": 20,
}

# ==============================================================================
# 2. Các hàm tiện ích (Không thay đổi)
# ==============================================================================
def sample_uniform_sphere(n_samples, dim, device=DEVICE):
    eta = torch.randn(n_samples, dim, device=device)
    return F.normalize(eta, p=2, dim=1)

def mobius_reparam(eps, mu, rho):
    rho = rho.unsqueeze(-1) if rho.dim() == 1 else rho
    eps_mu_dot = torch.sum(eps * mu, dim=1, keepdim=True)
    numerator = (1 - rho**2) * eps + 2 * rho**2 * mu + 2 * rho * eps_mu_dot * mu
    denominator = 1 + 2 * rho * eps_mu_dot + rho**2
    z = numerator / (denominator + config["epsilon"])
    return F.normalize(z, p=2, dim=1)

def rbf_kernel(x, y, sigma):
    dist_sq = 2 - 2 * (x @ y.t())
    return torch.exp(-dist_sq / (2 * sigma**2 + config["epsilon"]))

def mmd_loss(q_samples, p_samples, sigma=None):
    if q_samples.shape[0] < 2 or p_samples.shape[0] < 2:
        return torch.tensor(0.0, device=DEVICE)
    if sigma is None:
        with torch.no_grad():
            dists = torch.pdist(torch.cat([q_samples, p_samples], dim=0))
            sigma = dists.median()
    k_qq = rbf_kernel(q_samples, q_samples, sigma).mean()
    k_pp = rbf_kernel(p_samples, p_samples, sigma).mean()
    k_qp = rbf_kernel(q_samples, p_samples, sigma).mean()
    return k_qq + k_pp - 2 * k_qp

# ==============================================================================
# 3. Kiến trúc Mô hình (Không thay đổi)
# ==============================================================================
class EncoderCNN(nn.Module):
    def __init__(self, latent_dim):
        super(EncoderCNN, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
        )
        self.fc_block = nn.Sequential(nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(True))
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_s = nn.Linear(256, 1)
    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return self.fc_mu(x), self.fc_s(x)

class DecoderCNN(nn.Module):
    def __init__(self, latent_dim):
        super(DecoderCNN, self).__init__()
        self.fc_block = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(True), nn.Linear(256, 128 * 4 * 4), nn.ReLU(True))
        self.deconv_block = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1), nn.Sigmoid()
        )
    def forward(self, z):
        x = self.fc_block(z)
        x = x.view(-1, 128, 4, 4)
        return self.deconv_block(x)

class SphericalWAE_Supervised(nn.Module):
    def __init__(self, latent_dim, n_classes):
        super(SphericalWAE_Supervised, self).__init__()
        self.encoder = EncoderCNN(latent_dim)
        self.decoder = DecoderCNN(latent_dim)
        self.prior_mus = nn.Parameter(torch.randn(n_classes, latent_dim))
        self.rho_p = config["rho_prior"]
    def encode_to_distribution(self, x):
        mu_q_unnormalized, s_q = self.encoder(x)
        mu_q = F.normalize(mu_q_unnormalized, p=2, dim=1)
        rho_q = torch.sigmoid(s_q).squeeze(-1) * (1 - config["epsilon"])
        return mu_q, rho_q
    def forward(self, x):
        mu_q, rho_q = self.encode_to_distribution(x)
        eps = sample_uniform_sphere(x.shape[0], config["latent_dim"], device=x.device)
        z_q = mobius_reparam(eps, mu_q, rho_q)
        x_hat = self.decoder(z_q)
        return x_hat, z_q

# ==============================================================================
# 4. Hàm tính toán Mất mát (<<< PHẦN CHỈNH SỬA CHÍNH)
# ==============================================================================
def calculate_loss(x, y, x_hat, z_q, model, loss_fn_vgg, sup_mmd_weight, unsup_mmd_weight):
    # --- Thành phần Tái tạo ---
    bce_loss = F.binary_cross_entropy(x_hat, x, reduction='mean')

    # Chuyển đổi thang đo của ảnh từ [0, 1] sang [-1, 1] cho LPIPS
    x_rescaled = (x * 2) - 1
    x_hat_rescaled = (x_hat * 2) - 1
    # LPIPS yêu cầu ảnh 3 kênh, ta lặp lại kênh màu xám 3 lần
    x_rescaled_rgb = x_rescaled.repeat(1, 3, 1, 1)
    x_hat_rescaled_rgb = x_hat_rescaled.repeat(1, 3, 1, 1)

    lpips_loss = loss_fn_vgg(x_hat_rescaled_rgb, x_rescaled_rgb).mean()

    # Kết hợp hai loss tái tạo
    recon_loss = config["bce_weight"] * bce_loss + config["lpips_weight"] * lpips_loss

    # --- Các thành phần MMD (giữ nguyên) ---
    supervised_mmd_loss = 0.0
    normalized_prior_mus = F.normalize(model.prior_mus, p=2, dim=1)
    for c in range(config["n_classes"]):
        class_mask = (y == c)
        if class_mask.sum() > 1:
            supervised_mmd_loss += mmd_loss(z_q[class_mask],
                                            mobius_reparam(sample_uniform_sphere(class_mask.sum(), config["latent_dim"]),
                                                           normalized_prior_mus[c].expand(class_mask.sum(), -1),
                                                           torch.full((class_mask.sum(),), model.rho_p, device=DEVICE)))
    supervised_mmd_loss /= config["n_classes"]

    random_classes = torch.randint(0, config["n_classes"], (x.size(0),), device=DEVICE)
    z_p_unsupervised = mobius_reparam(sample_uniform_sphere(x.size(0), config["latent_dim"]),
                                      normalized_prior_mus[random_classes],
                                      torch.full((x.size(0),), model.rho_p, device=DEVICE))
    unsupervised_mmd_loss = mmd_loss(z_q, z_p_unsupervised)

    # --- Loss tổng hợp ---
    total_loss = recon_loss + \
                 (sup_mmd_weight * supervised_mmd_loss) + \
                 (unsup_mmd_weight * unsupervised_mmd_loss)

    return total_loss, recon_loss, supervised_mmd_loss, unsupervised_mmd_loss

# ==============================================================================
# 5. Vòng lặp Huấn luyện
# ==============================================================================
def train_epoch(model, train_loader, optimizer, epoch, scheduler, loss_fn_vgg): # <<< THAY ĐỔI: Thêm loss_fn_vgg
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['epochs']}")
    loss_acc, recon_acc, sup_mmd_acc, unsup_mmd_acc = 0.0, 0.0, 0.0, 0.0

    anneal_rate = min(1.0, (epoch + 1) / config["anneal_epochs"])
    current_sup_weight = config["sup_mmd_weight"] * anneal_rate
    current_unsup_weight = config["unsup_mmd_weight"] * anneal_rate

    for data, labels in pbar:
        data, labels = data.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        x_hat, z_q = model(data)

        # <<< THAY ĐỔI: Truyền loss_fn_vgg vào hàm loss
        loss, recon, sup_mmd, unsup_mmd = calculate_loss(data, labels, x_hat, z_q, model, loss_fn_vgg, current_sup_weight, current_unsup_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        loss_acc += loss.item()
        recon_acc += recon.item()
        sup_mmd_acc += sup_mmd.item()
        unsup_mmd_acc += unsup_mmd.item()

        pbar.set_postfix({
            "Loss": f"{loss.item():.3f}", "Recon": f"{recon.item():.3f}",
            "SupMMD": f"{sup_mmd.item():.4f}", "UnsupMMD": f"{unsup_mmd.item():.4f}",
            "λ": f"{current_sup_weight:.2f}", "γ": f"{current_unsup_weight:.2f}"
        })

    scheduler.step()
    n_batches = len(train_loader)
    print(f"====> Epoch {epoch+1} Avg Loss: Total={loss_acc/n_batches:.4f}, Recon={recon_acc/n_batches:.4f}, SupMMD={sup_mmd_acc/n_batches:.4f}, UnsupMMD={unsup_mmd_acc/n_batches:.4f}")
    return loss_acc/n_batches, recon_acc/n_batches, sup_mmd_acc/n_batches, unsup_mmd_acc/n_batches

# ==============================================================================
# 6. Trực quan hóa và Hàm chính
# ==============================================================================
def plot_random_samples_from_priors(model, save_dir="."):
    print("Bắt đầu sinh ảnh ngẫu nhiên từ các tiên nghiệm...")
    model.eval()
    n_classes, latent_dim = config["n_classes"], config["latent_dim"]
    fig, axes = plt.subplots(n_classes, 8, figsize=(8 * 1.5, n_classes * 1.5))
    with torch.no_grad():
        normalized_prior_mus = F.normalize(model.prior_mus, p=2, dim=1)
        for c in range(n_classes):
            mu_p = normalized_prior_mus[c].expand(8, -1)
            rho_p = torch.full((8,), model.rho_p, device=DEVICE)
            eps = sample_uniform_sphere(8, latent_dim, device=DEVICE)
            z_p = mobius_reparam(eps, mu_p, rho_p)
            generated_images = model.decoder(z_p)
            for i, img in enumerate(generated_images):
                ax = axes[c, i]
                ax.imshow(img.cpu().squeeze(), cmap='gray')
                ax.axis('off')
                if i == 0:
                    ax.text(-10, 14, f'{c}', verticalalignment='center', horizontalalignment='right', fontsize=12, fontweight='bold')
    plt.suptitle("Ảnh sinh ngẫu nhiên từ các thành phần Tiên nghiệm (Annealing)")
    plt.savefig(f'{save_dir}/random_samples_from_priors.png')
    plt.close(fig)

def from_latent(net, vec):
    with torch.no_grad():
        net.eval()
        vec_tensor = torch.from_numpy(vec).unsqueeze(0).to(DEVICE).float()
        vec_tensor_normalized = F.normalize(vec_tensor, p=2, dim=1)
        return net.decoder(vec_tensor_normalized).cpu().numpy().reshape(28, 28)

def get_sampling_grid(net, grid):
    base = torch.randn(config["latent_dim"] - 2)
    image_list = [torch.from_numpy(from_latent(net, np.hstack([vec_2d, base.numpy()]))) for vec_2d in grid]
    return torch.stack(image_list).unsqueeze(1)

def plot_grid_samples(model, save_dir="."):
    print("Bắt đầu sinh ảnh từ lưới phẳng (phương pháp so sánh)...")
    grid_points = list(product(np.linspace(-1.5, 1.5, 8), np.linspace(-1.5, 1.5, 8)))
    results = get_sampling_grid(model, grid_points)
    fig = plt.figure(figsize=(10, 10))
    img_grid = make_grid(results, nrow=8)
    plt.imshow(img_grid.permute(1, 2, 0))
    plt.title("Ảnh sinh ra từ Lưới Phẳng (Grid Sampling)")
    plt.axis('off')
    plt.savefig(f'{save_dir}/grid_samples.png')
    plt.close(fig)

def plot_results(history, model, test_loader, save_dir="."):
    print("Bắt đầu vẽ biểu đồ và trực quan hóa kết quả...")
    fig = plt.figure(figsize=(12, 8))
    plt.plot([h[0] for h in history], label='Total Loss')
    plt.plot([h[1] for h in history], label='Reconstruction Loss')
    plt.plot([h[2] for h in history], label='Supervised MMD Loss')
    plt.plot([h[3] for h in history], label='Unsupervised MMD Loss')
    plt.title('Lịch sử Huấn luyện (WAE with Annealing)')
    plt.xlabel('Epoch'); plt.ylabel('Loss Value'); plt.legend(); plt.grid(True)
    plt.savefig(f'{save_dir}/loss_history.png'); plt.close(fig)

    model.eval()
    with torch.no_grad():
        data, _ = next(iter(test_loader)); data = data.to(DEVICE); x_hat, _ = model(data)
        fig = plt.figure(figsize=(20, 4))
        n = 10
        for i in range(n):
            ax = plt.subplot(2, n, i + 1); plt.imshow(data[i].cpu().squeeze(), cmap='gray'); plt.title("Gốc"); ax.axis('off')
            ax = plt.subplot(2, n, i + 1 + n); plt.imshow(x_hat[i].cpu().squeeze(), cmap='gray'); plt.title("Tái tạo"); ax.axis('off')
        plt.savefig(f'{save_dir}/reconstructions.png'); plt.close(fig)

        print("Bắt đầu chiếu không gian ẩn bằng UMAP...")
        latent_mus, labels = [], []
        for data, target in tqdm(test_loader, desc="Encoding test set for UMAP"):
            mu_q, _ = model.encode_to_distribution(data.to(DEVICE))
            latent_mus.append(mu_q.cpu().numpy()); labels.append(target.numpy())

        latent_mus = np.concatenate(latent_mus, axis=0); labels = np.concatenate(labels, axis=0)

        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
        embedding = reducer.fit_transform(latent_mus)

        prior_mus_np = F.normalize(model.prior_mus, p=2, dim=1).cpu().detach().numpy()
        prior_embedding = reducer.transform(prior_mus_np)

        fig = plt.figure(figsize=(12, 10))
        scatter = plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap='Spectral', s=5, alpha=0.7)
        plt.scatter(prior_embedding[:, 0], prior_embedding[:, 1], c=range(10), cmap='Spectral', marker='*', s=500, edgecolor='black', label='Prior Centers')
        plt.title('Không gian ẩn (Latent Space) - UMAP (Annealing)')
        plt.legend(handles=scatter.legend_elements(num=10)[0], labels=list(range(10)))
        plt.colorbar(scatter); plt.savefig(f'{save_dir}/latent_space_umap.png'); plt.close(fig)

    plot_random_samples_from_priors(model, save_dir=save_dir)
    plot_grid_samples(model, save_dir=save_dir)

2025-08-15 09:04:29.076447: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755248669.246807      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755248669.294800      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

num_workers = 2 if os.name == 'nt' else 4
train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True, pin_memory=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, pin_memory=True, num_workers=num_workers)

model = SphericalWAE_Supervised(latent_dim=config["latent_dim"], n_classes=config["n_classes"]).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=config["lr"])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

# <<< THAY ĐỔI: Khởi tạo LPIPS loss function
loss_fn_vgg = lpips.LPIPS(net='vgg').to(DEVICE)

print(f"Bắt đầu huấn luyện trên thiết bị: {DEVICE} với kiến trúc WAE, Annealing và LPIPS Loss")
print(f"Cấu hình: {config}")

history = []
for epoch in range(config["epochs"]):
    # <<< THAY ĐỔI: Truyền loss_fn_vgg vào train_epoch
    avg_losses = train_epoch(model, train_loader, optimizer, epoch, scheduler, loss_fn_vgg)
    history.append(avg_losses)

print("Hoàn tất huấn luyện!")

torch.save(model.state_dict(), f'{SAVE_DIR}/spcauchy_wae_annealing_lpips.pth')
print(f"Đã lưu mô hình đã huấn luyện vào '{SAVE_DIR}/spcauchy_wae_annealing_lpips.pth'")

plot_results(history, model, test_loader, save_dir=SAVE_DIR)

100%|██████████| 9.91M/9.91M [00:00<00:00, 34.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.06MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.95MB/s]


Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 227MB/s]  


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth
Bắt đầu huấn luyện trên thiết bị: cuda với kiến trúc WAE, Annealing và LPIPS Loss
Cấu hình: {'latent_dim': 32, 'n_classes': 10, 'batch_size': 128, 'epochs': 50, 'lr': 0.001, 'rho_prior': 0.7, 'epsilon': 1e-08, 'bce_weight': 0.3, 'lpips_weight': 0.7, 'sup_mmd_weight': 20.0, 'unsup_mmd_weight': 50.0, 'anneal_epochs': 20}


Epoch 1/50: 100%|██████████| 469/469 [00:40<00:00, 11.55it/s, Loss=0.226, Recon=0.110, SupMMD=0.0980, UnsupMMD=0.0075, λ=1.00, γ=2.50]


...

====> Epoch 49 Avg Loss: Total=1.7886, Recon=0.0626, SupMMD=0.0738, UnsupMMD=0.0050


Epoch 50/50: 100%|██████████| 469/469 [00:40<00:00, 11.62it/s, Loss=2.167, Recon=0.061, SupMMD=0.0947, UnsupMMD=0.0043, λ=20.00, γ=50.00]


====> Epoch 50 Avg Loss: Total=1.7860, Recon=0.0623, SupMMD=0.0736, UnsupMMD=0.0050
Hoàn tất huấn luyện!
Đã lưu mô hình đã huấn luyện vào 'results_wae_annealing_v4_lpips/spcauchy_wae_annealing_lpips.pth'
Bắt đầu vẽ biểu đồ và trực quan hóa kết quả...
Bắt đầu chiếu không gian ẩn bằng UMAP...


Encoding test set for UMAP: 100%|██████████| 79/79 [00:00<00:00, 105.82it/s]
/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Bắt đầu sinh ảnh ngẫu nhiên từ các tiên nghiệm...
Bắt đầu sinh ảnh từ lưới phẳng (phương pháp so sánh)...


In [4]:
# ==============================================================================
# Sửa lỗi và Bổ sung hàm vẽ Slerp
# ==============================================================================

def slerp(p0, p1, t, epsilon=1e-8):
    """
    Nội suy trên mặt cầu (Spherical Linear Interpolation).
    p0, p1: các vector bắt đầu và kết thúc (đã được chuẩn hóa).
    t: một giá trị hoặc tensor chứa các giá trị từ 0 đến 1.
    """
    # Tính góc giữa hai vector
    omega = torch.acos(torch.dot(p0, p1).clamp(-1, 1))
    sin_omega = torch.sin(omega)

    # Nếu hai vector quá gần nhau, trả về vector ban đầu để tránh chia cho 0
    if sin_omega.item() < epsilon:
        # Mở rộng p0 để có cùng số chiều với output mong muốn khi t là một tensor
        return p0.unsqueeze(0).expand(len(t), -1)

    # Đảm bảo t có cùng device với các vector
    t = t.to(p0.device)

    # Công thức Slerp
    a = torch.sin((1.0 - t) * omega) / sin_omega
    b = torch.sin(t * omega) / sin_omega

    # unsqueeze() để thực hiện phép nhân broadcast đúng cách
    return a.unsqueeze(-1) * p0.unsqueeze(0) + b.unsqueeze(-1) * p1.unsqueeze(0)


def plot_slerp(model, save_dir=".", num_steps=10):
    """
    Vẽ và lưu ảnh được tạo ra từ phép nội suy Slerp giữa các cặp tiên nghiệm.
    """
    print("Bắt đầu sinh ảnh nội suy Slerp...")
    model.eval()

    # Chọn một vài cặp chữ số thú vị để nội suy
    interpolation_pairs = [(1, 7), (3, 5), (2, 8), (4, 9)]
    num_pairs = len(interpolation_pairs)

    fig, axes = plt.subplots(num_pairs, num_steps, figsize=(num_steps * 1.5, num_pairs * 1.5))

    with torch.no_grad():
        t_values = torch.linspace(0, 1, num_steps)

        for row, (digit_a, digit_b) in enumerate(interpolation_pairs):
            # LẤY VECTOR VÀ SỬA LỖI: Thêm `dim=0`
            mu_a = F.normalize(model.prior_mus[digit_a].detach(), dim=0)
            mu_b = F.normalize(model.prior_mus[digit_b].detach(), dim=0)

            # Thực hiện Slerp
            interpolated_z = slerp(mu_a, mu_b, t_values).to(DEVICE)

            # Giải mã các vector z trung gian
            generated_images = model.decoder(interpolated_z)

            # Vẽ các ảnh
            for col, img in enumerate(generated_images):
                ax = axes[row, col]
                ax.imshow(img.cpu().squeeze(), cmap='gray')
                ax.axis('off')

                # Ghi nhãn cho ảnh đầu và cuối
                if col == 0:
                    ax.set_title(f'{digit_a}')
                if col == num_steps - 1:
                    ax.set_title(f'{digit_b}')

    plt.suptitle("Nội suy Slerp giữa các Tiên nghiệm (Slerp Interpolation)")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Điều chỉnh layout để title không bị đè
    plt.savefig(f'{save_dir}/slerp_interpolation.png')
    plt.close(fig)
    print(f"Đã lưu ảnh Slerp vào '{save_dir}/slerp_interpolation.png'")

In [5]:
# THÊM LỆNH GỌI HÀM MỚI Ở ĐÂY
plot_slerp(model, save_dir=SAVE_DIR)

Bắt đầu sinh ảnh nội suy Slerp...
Đã lưu ảnh Slerp vào 'results_wae_annealing_v4_lpips/slerp_interpolation.png'


In [6]:
# <<< BƯỚC 1: Cài đặt thư viện cần thiết (nếu chưa có)
# !pip install pytorch-fid

import os
import torch
import torchvision
from tqdm import tqdm
from pytorch_fid import fid_score

# ==============================================================================
# PHẦN CẤU HÌNH (BẠN CẦN CHỈNH SỬA)
# ==============================================================================

# Đường dẫn đến file .pth chứa trọng số mô hình đã huấn luyện của bạn
MODEL_PATH = f'{SAVE_DIR}/spcauchy_wae_annealing_lpips.pth' 

# Số lượng ảnh để tính FID (khuyến nghị từ 10,000 đến 50,000 để có kết quả ổn định)
NUM_IMAGES_FOR_FID = 10000

# Tên thư mục để lưu ảnh
REAL_IMAGES_DIR = "fid_images_v2/real"
GEN_IMAGES_DIR = "fid_images_v2/generated"

# Tạo các thư mục nếu chúng chưa tồn tại
os.makedirs(REAL_IMAGES_DIR, exist_ok=True)
os.makedirs(GEN_IMAGES_DIR, exist_ok=True)

# ==============================================================================
# BƯỚC 2: LƯU ẢNH THẬT TỪ BỘ DỮ LIỆU TEST
# ==============================================================================
print(f"Bắt đầu lưu {NUM_IMAGES_FOR_FID} ảnh thật vào thư mục '{REAL_IMAGES_DIR}'...")

# Tải bộ dữ liệu test của MNIST
transform = transforms.Compose([
    transforms.Resize(28), # Đảm bảo kích thước ảnh phù hợp
    transforms.ToTensor(),
])
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False)

saved_count = 0
for images, _ in tqdm(test_loader, desc="Saving real images"):
    for i in range(images.size(0)):
        if saved_count >= NUM_IMAGES_FOR_FID:
            break
        # Lặp lại kênh màu xám 3 lần để tạo ảnh RGB giả
        image_rgb = images[i].repeat(3, 1, 1)
        torchvision.utils.save_image(image_rgb, os.path.join(REAL_IMAGES_DIR, f"real_{saved_count}.png"))
        saved_count += 1
    if saved_count >= NUM_IMAGES_FOR_FID:
        break

print(f"Đã lưu thành công {saved_count} ảnh thật.")

# ==============================================================================
# BƯỚC 3: TẢI MODEL VÀ LƯU ẢNH DO MODEL SINH RA
# ==============================================================================
print(f"Bắt đầu sinh và lưu {NUM_IMAGES_FOR_FID} ảnh giả vào thư mục '{GEN_IMAGES_DIR}'...")

# Tải lại kiến trúc model (đảm bảo nó khớp với file trọng số)
model = SphericalWAE_Supervised(
    latent_dim=config["latent_dim"], 
    n_classes=config["n_classes"]
).to(DEVICE)

# Tải trọng số đã huấn luyện
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval() # Chuyển model sang chế độ đánh giá

generated_count = 0
with torch.no_grad():
    while generated_count < NUM_IMAGES_FOR_FID:
        # Xác định số lượng ảnh cần sinh trong batch này
        num_to_gen = min(config["batch_size"], NUM_IMAGES_FOR_FID - generated_count)

        # Sinh ảnh bằng cách lấy mẫu từ các prior ngẫu nhiên
        random_classes = torch.randint(0, config["n_classes"], (num_to_gen,), device=DEVICE)
        normalized_prior_mus = F.normalize(model.prior_mus, p=2, dim=1)
        
        eps = sample_uniform_sphere(num_to_gen, config["latent_dim"], device=DEVICE)
        z_p = mobius_reparam(
            eps,
            normalized_prior_mus[random_classes],
            torch.full((num_to_gen,), model.rho_p, device=DEVICE)
        )
        
        generated_images = model.decoder(z_p)

        # Lưu các ảnh đã sinh ra
        for i in range(generated_images.size(0)):
            # Lặp lại kênh màu xám 3 lần để tạo ảnh RGB giả
            image_rgb = generated_images[i].cpu().repeat(3, 1, 1)
            torchvision.utils.save_image(image_rgb, os.path.join(GEN_IMAGES_DIR, f"gen_{generated_count}.png"))
            generated_count += 1
            if generated_count % 1000 == 0:
                print(f"... Đã sinh {generated_count}/{NUM_IMAGES_FOR_FID} ảnh")

print(f"Đã sinh và lưu thành công {generated_count} ảnh giả.")

# ==============================================================================
# BƯỚC 4: TÍNH TOÁN VÀ IN RA ĐIỂM FID
# ==============================================================================
print("\nBắt đầu tính toán điểm FID. Quá trình này có thể mất vài phút...")

fid_value = fid_score.calculate_fid_given_paths(
    paths=[REAL_IMAGES_DIR, GEN_IMAGES_DIR],
    batch_size=50,
    device=DEVICE,
    dims=2048  # Kích thước đặc trưng của InceptionV3
)

print("\n==============================================")
print(f"Điểm FID của mô hình là: {fid_value:.4f}")
print("==============================================")
print("(FID càng thấp, chất lượng ảnh sinh ra càng tốt)")

Bắt đầu lưu 10000 ảnh thật vào thư mục 'fid_images_v2/real'...


Saving real images:  99%|█████████▊| 78/79 [00:05<00:00, 13.62it/s]


Đã lưu thành công 10000 ảnh thật.
Bắt đầu sinh và lưu 10000 ảnh giả vào thư mục 'fid_images_v2/generated'...
... Đã sinh 1000/10000 ảnh
... Đã sinh 2000/10000 ảnh
... Đã sinh 3000/10000 ảnh
... Đã sinh 4000/10000 ảnh
... Đã sinh 5000/10000 ảnh
... Đã sinh 6000/10000 ảnh
... Đã sinh 7000/10000 ảnh
... Đã sinh 8000/10000 ảnh
... Đã sinh 9000/10000 ảnh
... Đã sinh 10000/10000 ảnh
Đã sinh và lưu thành công 10000 ảnh giả.

Bắt đầu tính toán điểm FID. Quá trình này có thể mất vài phút...


Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 325MB/s]
100%|██████████| 200/200 [00:42<00:00,  4.72it/s]



Điểm FID của mô hình là: 16.5517
(FID càng thấp, chất lượng ảnh sinh ra càng tốt)


In [7]:
# <<< BƯỚỚC 1: Cài đặt các thư viện cần thiết (nếu chưa có)
# !pip install scikit-learn torchmetrics scipy

import os
import torch
import numpy as np
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment # Dùng để tính ACC
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, accuracy_score

# TorchMetrics cho SSIM và PSNR
from torchmetrics.image import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio

# Tái sử dụng các phần đã có từ code của bạn
# (Giả định các lớp model, hàm, và config đã được định nghĩa ở trên)

# ==============================================================================
# PHẦN CẤU HÌNH (BẠN CẦN CHỈNH SỬA)
# ==============================================================================

# Đường dẫn đến file .pth chứa trọng số mô hình đã huấn luyện của bạn
MODEL_PATH = f'{SAVE_DIR}/spcauchy_wae_annealing_lpips.pth' 

# ==============================================================================
# BƯỚC 2: TẢI DỮ LIỆU VÀ MODEL
# ==============================================================================
print("Bắt đầu tải dữ liệu và mô hình...")

# Tải bộ dữ liệu test của MNIST (giữ nguyên từ code trước)
transform = transforms.Compose([transforms.ToTensor()])
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False)

# Tải lại kiến trúc model
model = SphericalWAE_Supervised(
    latent_dim=config["latent_dim"], 
    n_classes=config["n_classes"]
).to(DEVICE)

# Tải trọng số đã huấn luyện
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval() # Chuyển model sang chế độ đánh giá

print("Tải dữ liệu và mô hình thành công.")

# ==============================================================================
# BƯỚC 3: TÍNH METRICS TÁI TẠO (SSIM & PSNR)
# ==============================================================================
print("\n--- Bắt đầu tính toán Metrics Tái tạo ---")

# Khởi tạo các đối tượng metric từ torchmetrics
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(DEVICE)
psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(DEVICE)

with torch.no_grad():
    for images, _ in tqdm(test_loader, desc="Calculating SSIM & PSNR"):
        images = images.to(DEVICE)
        
        # Tái tạo lại ảnh
        reconstructed_images, _ = model(images)
        
        # Cập nhật giá trị cho các metric
        ssim_metric.update(reconstructed_images, images)
        psnr_metric.update(reconstructed_images, images)

# Lấy kết quả cuối cùng
final_ssim = ssim_metric.compute()
final_psnr = psnr_metric.compute()

print(f"Hoàn tất tính toán SSIM và PSNR.")
print(f"Kết quả SSIM: {final_ssim:.4f} (Càng gần 1 càng tốt)")
print(f"Kết quả PSNR: {final_psnr:.4f} (Càng cao càng tốt)")


# ==============================================================================
# BƯỚC 4: TÍNH METRICS PHÂN CỤM (ACC & NMI)
# ==============================================================================
print("\n--- Bắt đầu tính toán Metrics Phân cụm ---")

# Thu thập tất cả các vector ẩn và nhãn thật
all_latents = []
all_labels = []

print("Trích xuất các vector ẩn từ bộ dữ liệu test...")
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Encoding images"):
        images = images.to(DEVICE)
        
        # Mã hóa ảnh để lấy vector ẩn (sử dụng mu_q)
        mu_q, _ = model.encode_to_distribution(images)
        
        all_latents.append(mu_q.cpu().numpy())
        all_labels.append(labels.numpy())

# Gộp lại thành một mảng lớn
latents_np = np.concatenate(all_latents, axis=0)
labels_np = np.concatenate(all_labels, axis=0)

print(f"Đã trích xuất {len(latents_np)} vector ẩn. Bắt đầu chạy K-Means...")

# Chạy K-Means trên không gian ẩn
kmeans = KMeans(n_clusters=config["n_classes"], random_state=42, n_init='auto')
cluster_preds = kmeans.fit_predict(latents_np)

# --- Tính NMI ---
nmi_score = normalized_mutual_info_score(labels_np, cluster_preds)
print(f"Hoàn tất K-Means.")
print(f"Kết quả NMI: {nmi_score:.4f} (Càng gần 1 càng tốt)")

# --- Tính ACC ---
# Vì nhãn của K-Means (0,1,2,...) là ngẫu nhiên, ta cần tìm ánh xạ tối ưu
# giữa nhãn cụm và nhãn thật để tính ACC một cách chính xác.
# Chúng ta sử dụng thuật toán Hungarian để làm việc này.
contingency_matrix = np.zeros((config["n_classes"], config["n_classes"]), dtype=np.int64)
for i in range(len(labels_np)):
    contingency_matrix[cluster_preds[i], labels_np[i]] += 1

row_ind, col_ind = linear_sum_assignment(-contingency_matrix)
optimal_mapping = {row: col for row, col in zip(row_ind, col_ind)}
mapped_preds = np.array([optimal_mapping[pred] for pred in cluster_preds])

acc_score = accuracy_score(labels_np, mapped_preds)
print(f"Kết quả ACC: {acc_score:.4f} (Càng gần 1 càng tốt)")


# ==============================================================================
# BƯỚC 5: IN RA BẢNG TỔNG KẾT
# ==============================================================================
print("\n======================= BẢNG TỔNG KẾT METRICS =======================")
print(f"| Metric                      | Giá trị        | Diễn giải               |")
print(f"|-----------------------------|----------------|-------------------------|")
print(f"| FID (Fréchet Inception Dist.) | 34.1562        | Càng thấp càng tốt      |")
print(f"|-----------------------------|----------------|-------------------------|")
print(f"| SSIM (Structural Similarity)| {final_ssim:<14.4f} | Càng gần 1 càng tốt      |")
print(f"| PSNR (Peak Signal-to-Noise) | {final_psnr:<14.4f} | Càng cao càng tốt       |")
print(f"|-----------------------------|----------------|-------------------------|")
print(f"| NMI (Normalized Mutual Info)| {nmi_score:<14.4f} | Càng gần 1 càng tốt      |")
print(f"| ACC (Clustering Accuracy)   | {acc_score:<14.4f} | Càng gần 1 càng tốt      |")
print("=======================================================================")

Bắt đầu tải dữ liệu và mô hình...
Tải dữ liệu và mô hình thành công.

--- Bắt đầu tính toán Metrics Tái tạo ---


Calculating SSIM & PSNR: 100%|██████████| 79/79 [00:01<00:00, 66.03it/s]


Hoàn tất tính toán SSIM và PSNR.
Kết quả SSIM: 0.8719 (Càng gần 1 càng tốt)
Kết quả PSNR: 18.2050 (Càng cao càng tốt)

--- Bắt đầu tính toán Metrics Phân cụm ---
Trích xuất các vector ẩn từ bộ dữ liệu test...


Encoding images: 100%|██████████| 79/79 [00:00<00:00, 81.00it/s]

Đã trích xuất 10000 vector ẩn. Bắt đầu chạy K-Means...
Hoàn tất K-Means.
Kết quả NMI: 0.8907 (Càng gần 1 càng tốt)
Kết quả ACC: 0.9551 (Càng gần 1 càng tốt)

======================= BẢNG TỔNG KẾT METRICS =======================
| Metric                      | Giá trị        | Diễn giải               |
|-----------------------------|----------------|-------------------------|
| FID (Fréchet Inception Dist.) | 34.1562        | Càng thấp càng tốt      |
|-----------------------------|----------------|-------------------------|
| SSIM (Structural Similarity)| 0.8719         | Càng gần 1 càng tốt      |
| PSNR (Peak Signal-to-Noise) | 18.2050        | Càng cao càng tốt       |
|-----------------------------|----------------|-------------------------|
| NMI (Normalized Mutual Info)| 0.8907         | Càng gần 1 càng tốt      |
| ACC (Clustering Accuracy)   | 0.9551         | Càng gần 1 càng tốt      |
